# 18 · Extract per-slice features (for the slice aggregator)
Runs notebook 14's trained ResNet-18 over **all 24 slices** of every scan and saves, per slice:
- a **512-number feature vector** (the "notes" the ResNet writes before its final layer)
- the **12 logits** (its per-slice predictions)

The aggregator notebook (19) then trains a small model on these in minutes, with no images and no ResNet.

**Settings:** GPU on · Internet OFF is fine · Inputs: `02-preprocess-test-alex` (images + manifests) + notebook `14` output (weights).
Set `LIMIT = 20` for a quick CPU test, `LIMIT = None` for the real run.

In [ ]:
# Cell 1: setup
import os, glob, numpy as np, pandas as pd, torch, timm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
PLANES = ['sagittal', 'coronal', 'axial']
BASE_IN = '/kaggle/input/notebooks/alextanasescu/02-preprocess-test-alex'
OUT = '/kaggle/working'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LIMIT = None          # set to 20 for a quick test run on CPU; None = every scan

found = glob.glob('/kaggle/input/notebooks/*/*/resnet18_aug_15ep.pt')
print(DEVICE, found)
assert len(found) == 1, 'expected exactly one notebook-14 weights file'
WEIGHTS = found[0]

In [ ]:
# Cell 2: load the 3 manifests (images + split + the labels at the time; labels get re-joined later in 19)
mans = {}
for p in PLANES:
    m = pd.read_csv(f'{BASE_IN}/manifest_{p}.csv')
    m['path'] = f'{BASE_IN}/images_{p}/' + m['StudyInstanceUID'] + '.npy'
    m = m[['StudyInstanceUID', 'split', 'path'] + LABELS].reset_index(drop=True)
    if LIMIT:   # keep all val knees + a few train knees, so the check in Cell 6 still works
        m = pd.concat([m[m['split'] == 'val'], m[m['split'] == 'train'].head(LIMIT)]).reset_index(drop=True)
    mans[p] = m
    print(p, m['split'].value_counts().to_dict())

In [ ]:
# Cell 3: load notebook 14's model
model = timm.create_model('resnet18', pretrained=False, num_classes=12, in_chans=1)
model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
model = model.to(DEVICE).eval()
head = model.get_classifier()      # the final layer: 512 numbers -> 12 logits
print('model loaded; head:', head)

In [ ]:
# Cell 4: dataset that returns ALL 24 slices of a scan (same normalisation as training)
class VolumeDataset(Dataset):
    def __init__(self, df): self.df = df
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        vol = np.load(self.df['path'].iloc[i]).astype(np.float32) / 255.0   # (24, 224, 224)
        vol = (vol - 0.45) / 0.225
        return torch.from_numpy(vol).unsqueeze(1)                          # (24, 1, 224, 224)

@torch.no_grad()
def extract(df, batch_vols=8):
    loader = DataLoader(VolumeDataset(df), batch_size=batch_vols, shuffle=False, num_workers=4)
    feats, logits = [], []
    for i, x in enumerate(loader):                       # x: (B, 24, 1, 224, 224)
        B, S = x.shape[:2]
        x = x.view(B * S, 1, 224, 224).to(DEVICE)        # every slice becomes its own image
        with torch.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
            f = model.forward_head(model.forward_features(x), pre_logits=True)   # (B*24, 512)
            l = head(f)                                                         # (B*24, 12)
        feats.append(f.float().view(B, S, -1).cpu().numpy().astype(np.float16))
        logits.append(l.float().view(B, S, -1).cpu().numpy())
        if i % 100 == 0: print(f'  {i * batch_vols}/{len(df)}')
    return np.concatenate(feats), np.concatenate(logits)

In [ ]:
# Cell 5: run it for each plane and save
for p in PLANES:
    print('extracting', p)
    F, L = extract(mans[p])
    np.save(f'{OUT}/feats_{p}.npy', F)                 # (N, 24, 512) float16
    np.save(f'{OUT}/logits_{p}.npy', L)                # (N, 24, 12)
    mans[p].drop(columns='path').to_csv(f'{OUT}/ids_{p}.csv', index=False)   # row i of the arrays = row i here
    print(p, F.shape, L.shape)

In [ ]:
# Cell 6: sanity check - rebuild notebook 14's score from the saved logits.
# Mean of slices 6..18 per plane, then average planes per knee. Should land at ~0.789.
# (Slightly different from 0.789 is fine: this uses every slice 6-18, the submission used every 2nd.)
rows = []
for p in PLANES:
    L = np.load(f'{OUT}/logits_{p}.npy')
    ids = pd.read_csv(f'{OUT}/ids_{p}.csv')
    prob = 1 / (1 + np.exp(-L[:, 6:19]))               # sigmoid
    rows.append(pd.DataFrame(prob.mean(1), columns=LABELS).assign(StudyInstanceUID=ids['StudyInstanceUID'], split=ids['split']))
allp = pd.concat(rows)
val = allp[allp['split'] == 'val'].groupby('StudyInstanceUID')[LABELS].mean()
truth = pd.concat([mans[p] for p in PLANES]).drop_duplicates('StudyInstanceUID').set_index('StudyInstanceUID').loc[val.index, LABELS] >= 0.5
aucs = {l: roc_auc_score(truth[l], val[l]) for l in LABELS if truth[l].nunique() == 2}
print(f'knee-level AUC from saved logits: {np.mean(list(aucs.values())):.4f}  (expect ~0.79)')
print('files:', sorted(os.listdir(OUT)))